# RSNA Knee | DINOsaur V3 🦖


In [ ]:
from __future__ import annotations

import contextlib
import gc
import hashlib
import json
import math
import os
import re
import time
import warnings
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
import queue
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import cv2
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

@dataclass(frozen=True)
class RunSpec:
    time_limit_s: float = 8.65 * 3600
    dino2_batch: int = 1
    dino3_chunk: int = 48
    dino3_micro: int = 8
    rad_block: int = 32
    seed: int = 2026

SPEC = RunSpec()
START = time.time()
np.random.seed(SPEC.seed)
torch.manual_seed(SPEC.seed)
torch.backends.cudnn.benchmark = True

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)


def stamp(msg: str) -> None:
    print(f"[{time.time()-START:7.1f}s] {msg}", flush=True)


def release_device(device: torch.device | None = None) -> None:
    gc.collect()
    if not torch.cuda.is_available():
        return
    dev = DEVICE if device is None else device
    with torch.cuda.device(dev):
        torch.cuda.synchronize(dev)
        torch.cuda.empty_cache()


def release_cuda() -> None:
    release_device(DEVICE)


def find_competition_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
    ]
    for root in candidates:
        if (root / "test.csv").is_file() and (root / "test_series").is_dir():
            return root
    for test_csv in Path("/kaggle/input").glob("**/test.csv"):
        if (test_csv.parent / "test_series").is_dir() and (test_csv.parent / "test_series.csv").is_file():
            return test_csv.parent
    raise FileNotFoundError("Competition data was not found under /kaggle/input")

COMP = find_competition_root()
TEST = pd.read_csv(COMP / "test.csv", dtype={"StudyInstanceUID": str})
TEST_IDS = TEST["StudyInstanceUID"].astype(str).tolist()
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("This notebook requires a CUDA GPU")

stamp(f"competition={COMP}")
stamp(f"device={torch.cuda.get_device_name(0)}; gpus={torch.cuda.device_count()}")


def rank_columns(values: np.ndarray) -> np.ndarray:
    return pd.DataFrame(np.asarray(values, dtype=np.float64)).rank(
        axis=0, method="average", pct=True
    ).to_numpy(np.float64)


def normal_score(values: np.ndarray) -> np.ndarray:
    ranked = rank_columns(values)
    try:
        from scipy.special import ndtri
        n = max(len(ranked), 2)
        eps = 0.5 / (n + 1.0)
        return ndtri(np.clip(ranked, eps, 1.0 - eps))
    except Exception:
        return np.log(np.clip(ranked, 1e-6, 1-1e-6) / np.clip(1-ranked, 1e-6, 1))


def frame_from_scores(scores: np.ndarray, filename: str) -> pd.DataFrame:
    scores = np.asarray(scores, dtype=np.float64)
    if scores.shape != (len(TEST_IDS), len(LABELS)):
        raise ValueError(f"{filename}: unexpected prediction shape {scores.shape}")
    if not np.isfinite(scores).all():
        raise ValueError(f"{filename}: predictions contain NaN/Inf")
    out = pd.DataFrame(scores, columns=LABELS)
    out.insert(0, "StudyInstanceUID", TEST_IDS)
    out.to_csv(WORK / filename, index=False)
    return out


def sha256(path: Path, chunk: int = 8 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def locate_dir(name_hint: str, must_contain: str | None = None) -> Path:
    root = Path("/kaggle/input")
    exact = [p for p in root.glob("**/*") if p.is_dir() and p.name == name_hint]
    candidates = exact + [p for p in root.glob("**/*") if p.is_dir() and name_hint.lower() in p.name.lower()]
    for p in candidates:
        if must_contain is None or (p / must_contain).exists():
            return p
    raise FileNotFoundError(f"Could not locate input directory matching {name_hint!r}")


def checkpoint_metric_vector(obj: Any) -> np.ndarray | None:
    """Extract a genuine per-target validation metric vector when one is stored."""
    found: list[tuple[int, np.ndarray]] = []
    metric_tokens = ("auc", "metric", "score", "valid", "validation", "val_")
    reject_tokens = ("weight", "threshold", "preval", "prior", "alpha", "loss")

    def metric_path(path: str) -> bool:
        p = path.lower()
        return any(t in p for t in metric_tokens) and not any(t in p for t in reject_tokens)

    def accept(value: Any, path: str, priority: int):
        if not metric_path(path):
            return
        arr = None
        if isinstance(value, dict) and all(t in value for t in LABELS):
            try:
                arr = np.asarray([float(value[t]) for t in LABELS], np.float64)
            except Exception:
                arr = None
        elif isinstance(value, (list, tuple, np.ndarray)) and len(value) == len(LABELS):
            try:
                arr = np.asarray(value, np.float64)
            except Exception:
                arr = None
        if arr is not None and arr.shape == (len(LABELS),):
            if np.isfinite(arr).all() and ((arr >= 0.0) & (arr <= 1.0)).all() and float(np.std(arr)) > 1e-4:
                found.append((priority, arr))

    def walk(x: Any, path: str = ""):
        if isinstance(x, dict):
            accept(x, path, 3)
            for k, val in x.items():
                child = f"{path}/{k}"
                accept(val, child, 4 if "auc" in str(k).lower() else 2)
                if isinstance(val, (dict, list, tuple)):
                    walk(val, child)
        elif isinstance(x, (list, tuple)):
            for i, val in enumerate(x):
                if isinstance(val, (dict, list, tuple)):
                    walk(val, f"{path}/{i}")

    walk(obj)
    if not found:
        return None
    found.sort(key=lambda x: (x[0], float(np.std(x[1]))), reverse=True)
    return found[0][1]


def stable_softmax_weights(metric_matrix: np.ndarray | None, n_models: int, temperature: float = 0.025) -> np.ndarray:
    """Return [model,target] weights with strong shrinkage toward equal weighting."""
    if metric_matrix is None:
        return np.full((n_models, len(LABELS)), 1.0 / n_models, np.float64)
    m = np.asarray(metric_matrix, np.float64)
    if m.shape != (n_models, len(LABELS)) or not np.isfinite(m).all():
        return np.full((n_models, len(LABELS)), 1.0 / n_models, np.float64)
    centered = m - np.median(m, axis=0, keepdims=True)
    logits = np.clip(centered / max(temperature, 1e-6), -1.25, 1.25)
    raw = np.exp(logits - logits.max(axis=0, keepdims=True))
    raw /= raw.sum(axis=0, keepdims=True)
    equal = np.full_like(raw, 1.0 / n_models)
    return 0.70 * equal + 0.30 * raw


def weighted_rank_bank(rank_bank: np.ndarray, weights: np.ndarray) -> np.ndarray:
    bank = np.asarray(rank_bank, np.float64)
    w = np.asarray(weights, np.float64)
    if bank.ndim != 3 or w.shape != (bank.shape[0], bank.shape[2]):
        raise ValueError("rank bank / weight shape mismatch")
    return np.einsum("mnt,mt->nt", bank, w)


def robust_rank_consensus(rank_bank: np.ndarray) -> np.ndarray:
    bank = np.asarray(rank_bank, np.float64)
    mean = bank.mean(axis=0)
    median = np.median(bank, axis=0)
    ordered = np.sort(bank, axis=0)
    if len(bank) >= 5:
        trim = max(1, len(bank) // 10)
        trimmed = ordered[trim:-trim].mean(axis=0)
    else:
        trimmed = mean
    return 0.50 * mean + 0.30 * median + 0.20 * trimmed


def column_correlation(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a, b = np.asarray(a), np.asarray(b)
    out = np.zeros(a.shape[1], np.float64)
    for j in range(a.shape[1]):
        x, y = a[:, j], b[:, j]
        out[j] = 0.0 if x.std() < 1e-12 or y.std() < 1e-12 else float(np.corrcoef(x, y)[0, 1])
    return out


def fold_stability(rank_bank: np.ndarray) -> np.ndarray:
    bank = np.asarray(rank_bank, np.float64)
    m, _, t = bank.shape
    out = np.zeros(t, np.float64)
    pairs = 0
    for a in range(m):
        for b in range(a + 1, m):
            out += column_correlation(bank[a], bank[b])
            pairs += 1
    return out / max(pairs, 1)


def consistency_weighted_rank(rank_bank: np.ndarray, strength: float = 0.30):
    bank = np.asarray(rank_bank, np.float64)
    m, n, t = bank.shape
    equal = np.full((m, t), 1.0 / m, np.float64)

    # The tiny public example has only three studies, so correlation estimates
    # are deliberately disabled there. Hidden scoring has ~1300 studies.
    if n < 40 or m < 2:
        return bank.mean(axis=0), equal

    quality = np.zeros((m, t), np.float64)
    for model_index in range(m):
        peers = []
        for other_index in range(m):
            if other_index == model_index:
                continue
            peers.append(column_correlation(bank[model_index], bank[other_index]))
        quality[model_index] = np.mean(peers, axis=0)

    center = np.median(quality, axis=0, keepdims=True)
    scale = np.median(np.abs(quality - center), axis=0, keepdims=True) * 1.4826
    z = np.clip((quality - center) / np.maximum(scale, 0.015), -2.0, 2.0)
    raw = np.exp(z - z.max(axis=0, keepdims=True))
    raw /= raw.sum(axis=0, keepdims=True)
    weights = (1.0 - strength) * equal + strength * raw
    return weighted_rank_bank(bank, weights), weights


def robust_z(values: np.ndarray, floor: float = 0.02) -> np.ndarray:
    x = np.asarray(values, np.float64)
    med = float(np.median(x))
    mad = float(np.median(np.abs(x - med))) * 1.4826
    return (x - med) / max(mad, floor)


In [ ]:
# Shared DICOM helpers. The three model families intentionally keep their own
# sampling contracts, but they share safe low-level geometry and decoding routines.

_MULTI_SEP = re.compile(r"[_\-.]")
_FAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|water excit|\btirm\b|\bfatsup\b")
_FAT_OPTIONS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


def dicom_files(folder: Path) -> list[Path]:
    return sorted(folder.glob("*.dcm"), key=lambda p: p.name)


def header(path: Path, tags: Iterable[str] | None = None):
    try:
        return pydicom.dcmread(str(path), stop_before_pixels=True, force=True, specific_tags=list(tags) if tags else None)
    except Exception:
        return None


def vector(value: Any, n: int) -> np.ndarray | None:
    try:
        arr = np.asarray([float(x) for x in value], np.float64)
    except Exception:
        return None
    if len(arr) < n or not np.isfinite(arr[:n]).all():
        return None
    return arr[:n]


def image_center_x(ds) -> float | None:
    if ds is None:
        return None
    ipp = vector(getattr(ds, "ImagePositionPatient", None), 3)
    iop = vector(getattr(ds, "ImageOrientationPatient", None), 6)
    ps = vector(getattr(ds, "PixelSpacing", None), 2)
    try:
        rows, cols = float(ds.Rows), float(ds.Columns)
    except Exception:
        return None
    if ipp is None or iop is None or ps is None:
        return None
    center = ipp + iop[:3] * ps[1] * cols / 2.0 + iop[3:] * ps[0] * rows / 2.0
    return float(center[0])


def physical_order(folder: Path, fallback_dominant: bool = False) -> list[Path]:
    files = dicom_files(folder)
    rows = []
    for pos, path in enumerate(files):
        ds = header(path, ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
        ipp = vector(getattr(ds, "ImagePositionPatient", None), 3) if ds is not None else None
        iop = vector(getattr(ds, "ImageOrientationPatient", None), 6) if ds is not None else None
        inst = None
        try:
            inst = float(getattr(ds, "InstanceNumber", None))
        except Exception:
            pass
        proj = None
        if ipp is not None and iop is not None:
            proj = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        rows.append((path, ipp, inst, proj, pos))
    if rows and all(r[3] is not None for r in rows) and not fallback_dominant:
        return [r[0] for r in sorted(rows, key=lambda x: x[3])]
    if fallback_dominant:
        placed = [r for r in rows if r[1] is not None]
        if len(placed) >= max(2, int(0.8 * len(rows))):
            xyz = np.stack([r[1] for r in placed])
            axis = int(np.argmax(np.ptp(xyz, axis=0)))
            med = float(np.median(xyz[:, axis]))
            return [r[0] for r in sorted(rows, key=lambda x: (float(x[1][axis]) if x[1] is not None else med, x[2] if x[2] is not None else 1e12, x[4]))]
    if sum(r[2] is not None for r in rows) >= max(2, int(0.8 * len(rows))):
        return [r[0] for r in sorted(rows, key=lambda x: (x[2] if x[2] is not None else 1e12, x[4]))]
    return files


def decode_plane(path: Path) -> tuple[np.ndarray, Any] | tuple[None, None]:
    """Rescaled pixel contract used by the DINOv2 and RadImageNet checkpoints."""
    try:
        ds = pydicom.dcmread(str(path), force=True)
        arr = ds.pixel_array.astype(np.float32)
        arr = arr * float(getattr(ds, "RescaleSlope", 1) or 1)
        arr = arr + float(getattr(ds, "RescaleIntercept", 0) or 0)
        return arr, ds
    except Exception:
        return None, None


def decode_raw_plane(path: Path) -> tuple[np.ndarray, Any] | tuple[None, None]:
    """Raw stored-pixel contract used by the DINOv3 checkpoint family."""
    try:
        ds = pydicom.dcmread(str(path), force=True)
        return ds.pixel_array.astype(np.float32), ds
    except Exception:
        return None, None


def center_crop_mm(arr: np.ndarray, ds, mm: float, square_spacing: bool = True) -> np.ndarray:
    try:
        ps = vector(ds.PixelSpacing, 2)
    except Exception:
        ps = None
    if ps is None:
        return arr
    h, w = arr.shape[-2:]
    if square_spacing:
        hy = hx = int(round(mm / max(float(ps[0]), 1e-6) / 2.0))
    else:
        hy = int(round(mm / max(float(ps[0]), 1e-6) / 2.0))
        hx = int(round(mm / max(float(ps[1]), 1e-6) / 2.0))
    cy, cx = h // 2, w // 2
    y0, y1 = max(0, cy-hy), min(h, cy+hy)
    x0, x1 = max(0, cx-hx), min(w, cx+hx)
    crop = arr[y0:y1, x0:x1]
    return crop if crop.size else arr


def percentile_scale(volume: np.ndarray, lo_q: float = 1.0, hi_q: float = 99.0) -> np.ndarray:
    lo, hi = np.percentile(volume, [lo_q, hi_q])
    return np.clip((volume - lo) / max(float(hi-lo), 1e-6), 0.0, 1.0)


def resize_stack(volume: np.ndarray, size: int) -> np.ndarray:
    t = torch.from_numpy(np.ascontiguousarray(volume)).unsqueeze(0)
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False).squeeze(0)
    return t.mul(255).round().clamp(0, 255).to(torch.uint8).cpu().numpy()


In [ ]:
# ------------------------------- DINOv2 family -------------------------------

D2_SLOTS = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
D2_PRIOR = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
D2_FRONTIER = {
    "Fracture": "max", "Contusion": "max", "Medial Meniscus": "max",
    "Lateral Meniscus": "max", "ACL": "top2", "MCL": "top2", "Baker's": "max",
}

class D2AttentionHead(nn.Module):
    def __init__(self, in_dim: int, slots: int, outputs: int, use_prior: bool):
        super().__init__()
        hidden = 256
        self.proj = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(slots, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(outputs, hidden) * 0.02)
        self.drop = nn.Dropout(0.2)
        self.out = nn.Linear(hidden, outputs)
        self.hidden = hidden
        self.prior = bool(use_prior)
        if self.prior:
            p = torch.zeros(outputs, slots)
            for target, idxs in D2_PRIOR.items():
                p[LABELS.index(target), list(idxs)] = 0.55
            self.register_buffer("slot_prior", p)

    def forward(self, features, mask):
        h = self.proj(features) + self.slot_emb
        score = torch.einsum("bsh,oh->bos", h, self.query) / math.sqrt(self.hidden)
        if self.prior:
            score = score + self.slot_prior.unsqueeze(0)
        score = score.masked_fill(mask[:, None] < 0.5, -1e4)
        attn = score.softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", attn, h))
        return (ctx * self.out.weight[None]).sum(-1) + self.out.bias

class D2Network(nn.Module):
    def __init__(self, backbone, hidden: int, pool: str, prior: bool):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        parts = 3 if pool == "cls_mean_focal" else 2
        self.head = D2AttentionHead(hidden * parts, len(D2_SLOTS), len(LABELS), prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1))

    def forward(self, pixels, slot_mask, resize_to=None):
        b, s = pixels.shape[:2]
        x = pixels.reshape(b*s, *pixels.shape[2:]).float().div_(255.0)
        if resize_to is not None and x.shape[-1] != resize_to:
            x = F.interpolate(x, (resize_to, resize_to), mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        tokens = self.backbone(pixel_values=x).last_hidden_state
        patch = tokens[:, 1:]
        parts = [tokens[:,0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        return self.head(torch.cat(parts, 1).reshape(b, s, -1), slot_mask)


def _d2_manifest_root() -> Path:
    for m in Path("/kaggle/input").glob("**/manifest.json"):
        try:
            obj = json.loads(m.read_text())
            members = obj.get("members", [])
            if len(members) >= 15 and all((m.parent / str(x.get("file", ""))).is_file() for x in members):
                if any("pixel_group" in x for x in members):
                    return m.parent
        except Exception:
            pass
    raise FileNotFoundError("DINOv2 member package with manifest.json was not found")


def _d2_backbone_root() -> Path:
    for cfg in Path("/kaggle/input").glob("**/config.json"):
        try:
            obj = json.loads(cfg.read_text())
            if obj.get("model_type") == "dinov2" and int(obj.get("hidden_size", -1)) == 384:
                return cfg.parent
        except Exception:
            pass
    raise FileNotFoundError("Offline DINOv2-small model directory was not found")


def _d2_scan_series() -> pd.DataFrame:
    series_csv = pd.read_csv(COMP / "test_series.csv", dtype={"StudyInstanceUID":str, "SeriesInstanceUID":str})
    root = COMP / "test_series"
    jobs=[]
    for r in series_csv.itertuples(index=False):
        folder = root / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
        jobs.append((str(r.StudyInstanceUID), str(r.SeriesInstanceUID), str(r.Anatomical_Plane), folder))

    def inspect(job):
        study, series, plane, folder = job
        files = dicom_files(folder)
        ds = header(files[len(files)//2]) if files else None
        desc = f"{getattr(ds,'SeriesDescription','') or ''} {getattr(ds,'SequenceName','') or ''}".lower()
        desc = _MULTI_SEP.sub(" ", desc)
        options = getattr(ds, "ScanOptions", None) if ds is not None else None
        try:
            opt = list(options) if not isinstance(options, str) else re.split(r"[|\\]", options)
        except Exception:
            opt = []
        fs = bool(_FAT_RX.search(desc) or any(str(x).strip().upper() in _FAT_OPTIONS for x in opt))
        tr = float(getattr(ds, "RepetitionTime", np.nan) or np.nan) if ds is not None else np.nan
        te = float(getattr(ds, "EchoTime", np.nan) or np.nan) if ds is not None else np.nan
        scan = str(getattr(ds, "ScanningSequence", "") or "").upper() if ds is not None else ""
        if _T1_RX.search(desc) and not _T2_RX.search(desc) and not _PD_RX.search(desc): wt="T1"
        elif _T2_RX.search(desc) and not _PD_RX.search(desc): wt="T2"
        elif _PD_RX.search(desc): wt="PD"
        elif "GR" in scan: wt="GRE"
        elif np.isfinite(tr) and tr < 800: wt="T1"
        elif np.isfinite(te) and te > 60: wt="T2"
        elif np.isfinite(tr) and tr >= 800: wt="PD"
        else: wt="UNK"
        laterality = str(getattr(ds, "Laterality", "") or "").strip().upper()[:1] if ds is not None else ""
        laterality = laterality if laterality in ("L","R") else None
        cx = image_center_x(ds)
        corner_x = None
        try:
            corner_x = float(getattr(ds,"ImagePositionPatient")[0])
        except Exception:
            pass
        px = None
        try:
            px = float(getattr(ds, "PixelSpacing")[0])
        except Exception:
            pass
        return dict(study=study, series=series, plane=plane, folder=folder, files=files,
                    n=len(files), fatsat=fs, weight=wt, fluid=wt in ("PD","T2"),
                    laterality=laterality, center_x=cx, corner_x=corner_x, px=px)

    with ThreadPoolExecutor(max_workers=min(16, os.cpu_count() or 4)) as pool:
        rows=list(pool.map(inspect, jobs))
    return pd.DataFrame(rows)


def _d2_choose_slots(table: pd.DataFrame, allow_t1_fallback: bool) -> dict[str, dict[str, dict]]:
    out={uid:{} for uid in TEST_IDS}
    for uid, g in table.groupby("study"):
        for name, plane, fluid, fs in D2_SLOTS:
            q = g[(g.plane==plane) & (g.fatsat==fs) & (g.fluid==fluid)]
            if q.empty and allow_t1_fallback and fluid is False:
                q = g[(g.plane==plane) & (~g.fatsat)]
            if not q.empty:
                row = q.sort_values("n", ascending=False).iloc[0]
                out[str(uid)][name]=row.to_dict()
    return out


def _d2_side_map(table: pd.DataFrame, legacy_corner: bool) -> dict[str, str|None]:
    result={}
    for uid,g in table.groupby("study"):
        tags=[x for x in g.laterality.tolist() if x in ("L","R")]
        if tags:
            result[str(uid)] = tags[0]
            continue
        vals=[x for x in (g.corner_x.tolist() if legacy_corner else g.center_x.tolist()) if x is not None and np.isfinite(x)]
        if not vals:
            result[str(uid)] = None
            continue
        x=float(np.median(vals)); dead=5.0 if legacy_corner else 20.0
        result[str(uid)] = None if abs(x)<dead else ("R" if x<0 else "L")
    return result


def _d2_order_paths(rec: dict, legacy: bool) -> list[Path]:
    files = list(rec.get("files") or dicom_files(Path(rec["folder"])))
    if not files:
        return []
    if legacy:
        rows = []
        coords = []
        for order, path in enumerate(files):
            ds = header(path, ["ImagePositionPatient", "InstanceNumber"])
            ipp = vector(getattr(ds, "ImagePositionPatient", None), 3) if ds is not None else None
            inst = None
            try: inst = float(getattr(ds, "InstanceNumber", None))
            except Exception: pass
            if ipp is not None: coords.append(ipp)
            rows.append((path, ipp, inst, order))
        if len(coords) >= 2:
            axis = int(np.argmax(np.ptp(np.stack(coords), axis=0)))
            med = float(np.median(np.stack(coords)[:, axis]))
            return [r[0] for r in sorted(rows, key=lambda r: (float(r[1][axis]) if r[1] is not None else med, r[2] if r[2] is not None else 1e12, r[3]))]
        return files
    keyed = []
    for path in files:
        ds = header(path, ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"])
        key = None
        if ds is not None:
            ipp = vector(getattr(ds, "ImagePositionPatient", None), 3)
            iop = vector(getattr(ds, "ImageOrientationPatient", None), 6)
            if ipp is not None and iop is not None:
                key = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
            if key is None:
                try: key = float(ds.InstanceNumber)
                except Exception: pass
        keyed.append((key, path))
    if any(k is None for k, _ in keyed):
        return files
    return [path for key, path in sorted(keyed, key=lambda z: z[0])]


def _d2_read_slot(rec: dict, cfg: dict) -> np.ndarray | None:
    rules = cfg.get("rules") or {}
    legacy_order = rules.get("order") == "dominant_axis"
    files = _d2_order_paths(rec, legacy_order)
    n=len(files); count=int(cfg["slices"]); size=int(cfg["img"])
    if n == 0: return None
    lo_b, hi_b = map(float, cfg["band"])
    lo, hi = int(lo_b*(n-1)), int(hi_b*(n-1))
    idx=np.unique(np.linspace(lo, hi, count).astype(int)) if hi>lo else np.array([n//2])
    while len(idx)<count: idx=np.append(idx, idx[-1])
    planes=[]; datasets=[]
    for i in idx[:count]:
        a,ds=decode_plane(files[int(i)])
        planes.append(a); datasets.append(ds)
    valid=[i for i,a in enumerate(planes) if a is not None]
    if not valid: return None
    if rules.get("decode_fill") == "zero":
        shape = planes[valid[0]].shape
        planes=[np.zeros(shape,np.float32) if a is None else a for a in planes]
    else:
        for i,a in enumerate(planes):
            if a is None: planes[i]=planes[min(valid,key=lambda j:abs(j-i))]
    shape=planes[0].shape
    planes=[a if a.shape==shape else np.zeros(shape,np.float32) for a in planes]
    vol=np.stack(planes)
    px=rec.get("px")
    if px and np.isfinite(px) and px>0:
        want=int(round(float(cfg["crop_mm"])/float(px)))
        h,w=shape
        if 16 < want < min(h,w):
            cy,cx=h//2,w//2; half=want//2
            vol=vol[:, max(0,cy-half):cy+half, max(0,cx-half):cx+half]
    vol=percentile_scale(vol)
    return resize_stack(vol,size)


def _d2_cache(table: pd.DataFrame, cfg: dict) -> tuple[list[str], np.ndarray, np.ndarray]:
    rules=cfg.get("rules") or {}
    slots=_d2_choose_slots(table, allow_t1_fallback=bool(rules.get("slot_fallback",False)))
    sides=_d2_side_map(table, legacy_corner=rules.get("lat")=="corner_x")
    ids=list(TEST_IDS)
    arr=np.zeros((len(ids),len(D2_SLOTS),int(cfg["slices"]),int(cfg["img"]),int(cfg["img"])),np.uint8)
    mask=np.zeros((len(ids),len(D2_SLOTS)),np.float32)
    jobs=[]
    for i,uid in enumerate(ids):
        for s,(name,plane,_,_) in enumerate(D2_SLOTS):
            if name in slots.get(uid,{}): jobs.append((i,s,plane,slots[uid][name]))
    def job(x):
        i,s,plane,rec=x
        pix=_d2_read_slot(rec,cfg)
        if pix is not None and sides.get(ids[i])=="R":
            pix = pix[:, :, ::-1].copy() if plane in ("Coronal","Axial") else pix[::-1].copy()
        return i,s,pix
    with ThreadPoolExecutor(max_workers=min(12,os.cpu_count() or 4)) as pool:
        for i,s,pix in pool.map(job,jobs):
            if pix is not None:
                arr[i,s]=pix; mask[i,s]=1.0
    stamp(f"DINOv2 cache ready: {arr.shape}, slots={int(mask.sum())}")
    return ids,arr,mask


def _d2_build(cfg: dict):
    from transformers import AutoModel
    bb=AutoModel.from_pretrained(str(_d2_backbone_root()), local_files_only=True)
    pool=cfg.get("pool","cls_mean")
    return D2Network(bb, int(bb.config.hidden_size), pool, bool(cfg.get("prior",False)))


def _d2_frontier(prob_stack: torch.Tensor) -> torch.Tensor:
    out=prob_stack.mean(0)
    for target,mode in D2_FRONTIER.items():
        j=LABELS.index(target); x=prob_stack[:,:,j]
        if mode=="max": out[:,j]=x.max(0).values
        elif mode=="top2": out[:,j]=x.topk(min(2,x.shape[0]),dim=0).values.mean(0)
    return out

@torch.inference_mode()
@torch.inference_mode()
def _d2_verify_fingerprint(model, expected, size: int, group: int, member_id: str, device: torch.device):
    if expected is None:
        return
    gen = torch.Generator().manual_seed(2026)
    images = torch.randint(
        0, 256, (2, len(D2_SLOTS), group, size, size),
        generator=gen, dtype=torch.uint8
    ).to(device)
    mask = torch.ones(2, len(D2_SLOTS), device=device)
    mask[1, -1] = 0.0
    with torch.inference_mode():
        got = model(images, mask, size).float().cpu().numpy()
    ref = np.asarray(expected, np.float32)
    if got.shape != ref.shape:
        raise RuntimeError(
            f"DINOv2 {member_id}: fingerprint shape {got.shape} != {ref.shape}"
        )
    delta = float(np.max(np.abs(got - ref)))
    if delta > 0.002:
        raise RuntimeError(
            f"DINOv2 {member_id}: fingerprint mismatch {delta:.5f}"
        )


def _d2_fold_holdout_consensus(bank: np.ndarray, members: list[dict]) -> np.ndarray:
    by_fold = {}
    for i, info in enumerate(members):
        by_fold.setdefault(info.get("fold"), []).append(i)
    fold_predictions = []
    for fold, idxs in sorted(by_fold.items(), key=lambda kv: str(kv[0])):
        scores = np.asarray([float(members[i]["holdout"]) if members[i].get("holdout") is not None else np.nan for i in idxs], np.float64)
        if np.isfinite(scores).sum() < 2:
            w = np.full(len(idxs), 1.0 / len(idxs), np.float64)
        else:
            scores = np.where(np.isfinite(scores), scores, np.nanmedian(scores))
            z = np.clip((scores - scores.max()) / 0.006, -1.2, 0.0)
            soft = np.exp(z); soft /= soft.sum()
            equal = np.full(len(idxs), 1.0 / len(idxs), np.float64)
            w = 0.55 * equal + 0.45 * soft
        fold_predictions.append(np.tensordot(w, bank[idxs], axes=(0, 0)))
    return np.stack(fold_predictions).mean(axis=0)


def _d2_predict_member(model, pixels, mask, group, size, device: torch.device):
    starts = list(range(pixels.shape[2] - group + 1))
    out = np.zeros((len(pixels), len(LABELS)), np.float32)
    batch_size = max(1, int(SPEC.dino2_batch))

    cursor = 0
    while cursor < len(pixels):
        stop = min(len(pixels), cursor + batch_size)
        try:
            m = torch.from_numpy(
                np.ascontiguousarray(mask[cursor:stop])
            ).to(device, non_blocking=True)

            window_values = []
            for start in starts:
                x = torch.from_numpy(
                    np.ascontiguousarray(
                        pixels[cursor:stop, :, start:start + group]
                    )
                ).to(device, non_blocking=True)
                with torch.inference_mode(), torch.autocast(
                    "cuda", dtype=torch.float16
                ):
                    prob = torch.sigmoid(model(x, m, size).float())
                window_values.append(prob.cpu())
                del x, prob

            stacked = torch.stack(window_values, dim=0)
            out[cursor:stop] = _d2_frontier(stacked).numpy()
            del m, stacked, window_values
            cursor = stop

        except torch.cuda.OutOfMemoryError:
            release_device(device)
            if batch_size == 1:
                raise RuntimeError(
                    f"DINOv2 exhausted {device} even at batch=1"
                )
            batch_size = max(1, batch_size // 2)
            stamp(f"DINOv2 {device}: OOM retry with batch={batch_size}")

    return out


def run_dino2() -> dict:
    root = _d2_manifest_root()
    manifest = json.loads((root / "manifest.json").read_text())
    members = list(manifest["members"])
    table = _d2_scan_series()

    groups: dict[str, list[tuple[int, dict]]] = {}
    for original_index, member in enumerate(members):
        groups.setdefault(member["pixel_group"], []).append(
            (original_index, member)
        )

    # This path intentionally uses a single CUDA context. The same execution
    # pattern already completed all 20 DINOv2 members on Tesla T4.
    device = torch.device("cuda:0")
    rank_results: dict[int, np.ndarray] = {}
    metric_results: dict[int, np.ndarray | None] = {}
    info_results: dict[int, dict] = {}

    active_model = None
    active_arch = None

    for group_key, indexed_members in groups.items():
        cfg = json.loads(group_key)
        _, pixels, mask = _d2_cache(table, cfg)

        # Preserve manifest order so the exact equal-rank anchor is reproducible.
        for original_index, member in indexed_members:
            ck = torch.load(
                root / member["file"],
                map_location="cpu",
                weights_only=False,
            )
            mc = member.get("config", {})
            arch_key = (
                str(mc.get("variant", "small")),
                str(mc.get("pool", "cls_mean")),
                bool(mc.get("prior", False)),
            )

            if active_model is None or arch_key != active_arch:
                if active_model is not None:
                    del active_model
                    active_model = None
                    release_device(device)

                active_model = _d2_build({
                    **mc,
                    "pool": mc.get("pool", "cls_mean"),
                    "prior": bool(mc.get("prior", False)),
                }).to(device).eval()
                active_arch = arch_key

            active_model.load_state_dict(ck["model"], strict=True)

            _d2_verify_fingerprint(
                active_model,
                ck.get("fingerprint"),
                int(cfg["img"]),
                int(cfg["group"]),
                str(member.get("id")),
                device,
            )

            release_device(device)

            pred = _d2_predict_member(
                active_model,
                pixels,
                mask,
                int(cfg["group"]),
                int(cfg["img"]),
                device,
            )

            rank_results[original_index] = rank_columns(pred)
            metric_results[original_index] = checkpoint_metric_vector(ck)
            info_results[original_index] = {
                "id": member.get("id"),
                "fold": member.get("fold"),
                "holdout": member.get("holdout"),
            }

            stamp(
                f"DINOv2 {member.get('id')} "
                f"fold={member.get('fold')} done"
            )

            del ck, pred
            release_device(device)

        del pixels, mask
        gc.collect()
        release_device(device)

    if active_model is not None:
        del active_model
        release_device(device)

    if len(rank_results) != len(members):
        missing = sorted(set(range(len(members))) - set(rank_results))
        raise RuntimeError(
            f"DINOv2 missing member predictions: {missing}"
        )

    bank = np.stack([
        rank_results[index] for index in range(len(members))
    ])
    metrics = [
        metric_results[index] for index in range(len(members))
    ]
    member_info = [
        info_results[index] for index in range(len(members))
    ]

    metric_matrix = (
        None
        if any(item is None for item in metrics)
        else np.stack(metrics)
    )
    metric_weights = stable_softmax_weights(
        metric_matrix, len(bank)
    )

    equal = bank.mean(axis=0)
    metric_weighted = weighted_rank_bank(
        bank, metric_weights
    )
    holdout_weighted = _d2_fold_holdout_consensus(
        bank, member_info
    )
    robust = robust_rank_consensus(bank)
    consistent, consistency_weights = consistency_weighted_rank(
        bank, strength=0.18
    )

    # The verified public frontier remains dominant.
    enhanced = (
        0.60 * equal
        + 0.16 * holdout_weighted
        + 0.10 * robust
        + 0.08 * consistent
        + 0.06 * metric_weighted
    )

    return {
        "equal": equal,
        "weighted": metric_weighted,
        "holdout": holdout_weighted,
        "robust": robust,
        "consistent": consistent,
        "enhanced": enhanced,
        "bank": bank,
        "metrics": metric_matrix,
        "members": member_info,
        "root": root,
        "consistency_weights": consistency_weights,
    }


In [ ]:
# ------------------------------- DINOv3 family -------------------------------
# The checkpoint contract is recreated with an independently organized implementation.

D3_SLOTS=[("Sagittal",1),("Sagittal",0),("Coronal",1),("Coronal",0),("Axial",1),("Axial",0)]
D3_NSLICE=16; D3_SIZE=336; D3_CROP=130.0; D3_BAND=(0.12,0.88)
N_SLOT_TYPES=6; MASK_IDX=0
IMAGENET_MEAN=(0.485,0.456,0.406); IMAGENET_STD=(0.229,0.224,0.225)

class D3SlotToken(nn.Module):
    def __init__(self, vit, n_cat=6):
        super().__init__(); self.vit=vit
        self.tok=nn.Embedding(n_cat+1,vit.embed_dim,padding_idx=0)
        self.num_features=vit.num_features
        self._orig_prefix=getattr(vit,"num_prefix_tokens",1)
        vit.num_prefix_tokens=self._orig_prefix+1
        for block in vit.blocks:
            att=getattr(block,"attn",None)
            if att is not None and hasattr(att,"num_prefix_tokens"):
                att.num_prefix_tokens += 1
    def forward_features(self,x,cat):
        v=self.vit; x=v.patch_embed(x); pos=v._pos_embed(x); rope=None
        if isinstance(pos,tuple): x,rope=pos
        else: x=pos
        if getattr(v,"patch_drop",None) is not None: x=v.patch_drop(x)
        if getattr(v,"norm_pre",None) is not None: x=v.norm_pre(x)
        n=self._orig_prefix; x=torch.cat([x[:,:n],self.tok(cat).unsqueeze(1),x[:,n:]],1)
        if rope is not None:
            if getattr(v,"rope_mixed",False):
                for i,b in enumerate(v.blocks): x=b(x,rope=rope[i])
            else:
                for b in v.blocks: x=b(x,rope=rope)
        else: x=v.blocks(x)
        return v.norm(x)
    def forward_head(self,x,pre_logits=True): return self.vit.forward_head(x,pre_logits=pre_logits)

class _D3DepthBlock(nn.Module):
    def __init__(self,n_slice,dropout=0.0,ls_init=0.1):
        super().__init__(); self.norm=nn.GroupNorm(1,n_slice); self.v=nn.Conv2d(n_slice,n_slice,1); self.g=nn.Conv2d(n_slice,n_slice,1); self.out=nn.Conv2d(n_slice,n_slice,1); self.gamma=nn.Parameter(torch.full((n_slice,1,1),ls_init)); self.drop=nn.Dropout2d(dropout) if dropout else nn.Identity()
    def forward(self,x):
        z=self.norm(x); return x+self.gamma*self.drop(self.out(self.v(z)*F.silu(self.g(z))))

class D3Compress(nn.Module):
    def __init__(self,n_slice=16,out_ch=3):
        super().__init__(); self.imagenet=True; self.blocks=nn.ModuleList([_D3DepthBlock(n_slice)]); self.proj=nn.Conv2d(n_slice,out_ch,1,bias=True); self.register_buffer("mu",torch.tensor(IMAGENET_MEAN).view(1,-1,1,1)); self.register_buffer("sd",torch.tensor(IMAGENET_STD).view(1,-1,1,1))
    def forward(self,x):
        keep=(x.amax(1,keepdim=True)>0).to(x.dtype); z=x
        for b in self.blocks: z=b(z)
        z=self.proj(z); z=(z-self.mu.to(z.dtype))/self.sd.to(z.dtype)
        return z*keep

class D3Mixer(nn.Module):
    def __init__(self,n_slice=16,ksize=5,alpha_max=0.25):
        super().__init__(); self.n_slice=n_slice; self.ksize=ksize; self.r=ksize//2; self.alpha_max=alpha_max
        b=torch.tensor([1.0,4.0,6.0,4.0,1.0]); self.register_buffer("base",b.log()[self.r:]); u=self.r+1
        self.shared=nn.Parameter(torch.zeros(u)); self.plane_k=nn.Parameter(torch.zeros(3,u)); self.contrast_k=nn.Parameter(torch.zeros(2,u)); self.g0=nn.Parameter(torch.zeros(())); self.gate_p=nn.Parameter(torch.zeros(3)); self.gate_c=nn.Parameter(torch.zeros(2)); idx=torch.arange(n_slice); self.register_buffer("off",idx[None]-idx[:,None])
    def _pc(self,slot):
        s=torch.clamp(slot-1,0,5); return s//2,s%2
    def kernel(self,slot):
        p,c=self._pc(slot); half=self.base+self.shared+self.plane_k[p]+self.contrast_k[c]; full=torch.cat([half.flip(-1)[...,:self.r],half],-1); return F.softmax(full,-1)
    def alpha(self,slot):
        p,c=self._pc(slot); return self.alpha_max*torch.tanh(self.g0+self.gate_p[p]+self.gate_c[c])
    def forward(self,x,slot,vmask):
        t,s,h,w=x.shape
        if vmask is None: raise ValueError("DINOv3 mixer requires a valid-slice mask")
        k=self.kernel(slot); v=vmask.to(k.dtype); d=self.off+self.r; inside=(d>=0)&(d<self.ksize); kk=k[:,d.clamp(0,self.ksize-1)]*inside; M=kk*v[:,None,:]; den=M.sum(-1,keepdim=True); eye=torch.eye(s,device=x.device,dtype=M.dtype).expand(t,s,s); ok=(den>1e-6)&v[:,:,None].bool(); M=torch.where(ok,M/den.clamp_min(1e-6),eye); a=self.alpha(slot)[:,None,None]; op=((1-a)*eye+a*M).to(x.dtype); return torch.bmm(op,x.reshape(t,s,h*w)).reshape(t,s,h,w)


def d3_mean_max(v,sidx,b):
    d=v.shape[1]; cnt=torch.zeros(b,device=v.device,dtype=v.dtype).index_add_(0,sidx,torch.ones(len(v),device=v.device,dtype=v.dtype)); mean=torch.zeros(b,d,device=v.device,dtype=v.dtype).index_add_(0,sidx,v)/cnt.clamp_min(1).unsqueeze(1); mx=torch.full((b,d),-1e4,device=v.device,dtype=v.dtype); mx=mx.scatter_reduce(0,sidx[:,None].expand(-1,d),v,reduce="amax",include_self=True); return torch.cat([mean,mx],1)


def d3_pad_tokens(x,sidx,b,norm):
    t,p,d=x.shape; cnt=torch.bincount(sidx,minlength=b); s=int(cnt.max()); start=torch.cumsum(cnt,0)-cnt; pos=torch.arange(t,device=x.device)-start[sidx]; pad=x.new_zeros(b,s,p,d); pad[sidx,pos]=x; keep=torch.zeros(b,s,dtype=torch.bool,device=x.device); keep[sidx,pos]=True; return norm(pad.reshape(b,s*p,d)),~keep.repeat_interleave(p,dim=1)

class D3CodexPool(nn.Module):
    def __init__(self,d,n_labels=12,n_heads=6,pe=64,dropout=0.2):
        super().__init__(); self.q=nn.Parameter(torch.randn(n_labels,d)*0.02); self.kv_norm=nn.LayerNorm(d); self.attn=nn.MultiheadAttention(d,n_heads,dropout=dropout,batch_first=True); self.d_norm=nn.LayerNorm(d); self.dw=nn.Parameter(torch.randn(n_labels,d)*(1.0/math.sqrt(d))); self.db=nn.Parameter(torch.zeros(n_labels)); self.gate=nn.Parameter(torch.zeros(n_labels)); self.base=nn.Sequential(nn.LayerNorm(2*d+pe),nn.Dropout(dropout),nn.Linear(2*d+pe,n_labels))
    def forward(self,tok,slot,sidx,b,pres):
        base=self.base(torch.cat([d3_mean_max(tok[:,0],sidx,b),pres],1)); kv,kpm=d3_pad_tokens(tok[:,1:],sidx,b,self.kv_norm); q=self.q[None].expand(b,-1,-1); att=self.attn(q,kv,kv,key_padding_mask=kpm,need_weights=False)[0]; delta=(self.d_norm(att)*self.dw).sum(-1)+self.db; return base+self.gate*delta

class D3Readout(nn.Module):
    def __init__(self,d,pe=64):
        super().__init__(); self.pool_kind="xcodex"; self.k=len(LABELS); self.pres_emb=nn.Embedding(7,pe,padding_idx=0); self.pool=D3CodexPool(d,len(LABELS),pe=pe); self.drop=nn.Dropout(0.2)
    def forward(self,f,slot,sidx,b):
        pe=self.pres_emb(slot); pres=torch.zeros(b,pe.shape[1],device=f.device,dtype=f.dtype).index_add_(0,sidx,pe); return self.pool(f,slot,sidx,b,pres)

class D3Net(nn.Module):
    def __init__(self,enc,cond,n_meta=0,pool="xcodex",stem="native",n_slice=16):
        super().__init__(); self.enc=enc; self.cond=cond; self.compress=D3Compress(n_slice,3) if stem=="compress" else None; self.mixer=D3Mixer(n_slice) if stem=="mixer" else None; self.tokens=True; self.meta_mlp=nn.Sequential(nn.LayerNorm(n_meta),nn.Linear(n_meta,128),nn.GELU(),nn.Linear(128,enc.num_features)) if n_meta>0 else None; self.readout=D3Readout(enc.num_features); self.slot_emb=nn.Embedding(7,enc.num_features,padding_idx=0) if cond=="post" else None
    def forward(self,im,slot,smeta,sidx,b,vm=None):
        if self.mixer is not None: im=self.mixer(im,slot,vm)
        if self.compress is not None: im=self.compress(im)
        f=self.enc.forward_features(im,slot) if self.cond=="token" else self.enc.forward_features(im)
        inner=getattr(self.enc,"vit",self.enc); orig=getattr(self.enc,"_orig_prefix",getattr(inner,"num_prefix_tokens",1)); f=torch.cat([f[:,:1],f[:,orig:]],1)
        if self.cond=="post": f=f+self.slot_emb(slot)[:,None]
        if self.meta_mlp is not None and smeta.shape[1]>0: f=torch.cat([f,self.meta_mlp(smeta)[:,None]],1)
        return self.readout(f,slot,sidx,b)


def _d3_root() -> Path:
    for p in Path("/kaggle/input").glob("**/*"):
        if p.is_dir() and list(p.glob("*_f*.pt")):
            try:
                z=torch.load(sorted(p.glob("*_f*.pt"))[0],map_location="cpu",weights_only=False)
                if isinstance(z,dict) and "cfg" in z and "dinov3" in str(z["cfg"].get("backbone","")).lower(): return p
            except Exception: pass
    raise FileNotFoundError("DINOv3 fold checkpoint directory not found")


def _d3_series_data():
    ser=pd.read_csv(COMP/"test_series.csv",dtype={"StudyInstanceUID":str,"SeriesInstanceUID":str}); ser=ser.loc[:,~ser.columns.duplicated()]; by={s:g.to_dict("records") for s,g in ser[ser.StudyInstanceUID.isin(TEST_IDS)].groupby("StudyInstanceUID")}; return by


def _d3_order(folder: Path) -> list[Path]:
    rows=[]
    for f in dicom_files(folder):
        ds=header(f,["InstanceNumber"])
        try: key=int(ds.InstanceNumber)
        except Exception: continue
        rows.append((key,f))
        if len(rows)>=256: break
    return [f for _,f in sorted(rows)]


def _d3_side(path: Path) -> float:
    ds=header(path,["ImagePositionPatient"])
    try: return float(ds.ImagePositionPatient[0])
    except Exception: return 0.0


def _d3_render(path: Path, flip: bool) -> np.ndarray | None:
    arr, ds = decode_raw_plane(path)
    if arr is None:
        return None
    try:
        spacing = float(ds.PixelSpacing[0])
    except Exception:
        spacing = D3_CROP / max(arr.shape)
    half = int(round(D3_CROP / max(spacing, 1e-6) / 2.0))
    cy, cx = arr.shape[0] // 2, arr.shape[1] // 2
    crop = arr[max(0,cy-half):min(arr.shape[0],cy+half), max(0,cx-half):min(arr.shape[1],cx+half)]
    if crop.size == 0:
        return None
    lo, hi = np.percentile(crop[::4, ::4], [1, 99])
    image = np.clip((crop - lo) / max(float(hi-lo), 1e-6), 0.0, 1.0)
    image = cv2.resize(image, (D3_SIZE, D3_SIZE), interpolation=cv2.INTER_AREA)
    return image[:, ::-1].copy() if flip else image


def _d3_build_study(job):
    idx,uid,recs=job; out=np.zeros((6,D3_NSLICE,D3_SIZE,D3_SIZE),np.uint8); mask=np.zeros(6,np.uint8); rows=pd.DataFrame(recs)
    if len(rows):
        for slot,(plane,fs) in enumerate(D3_SLOTS):
            q=rows[(rows.Anatomical_Plane==plane)&(rows.Fat_Suppression==fs)]
            if q.empty: continue
            files=_d3_order(COMP/"test_series"/uid/str(q.iloc[0].SeriesInstanceUID))
            if not files: continue
            flip=plane!="Sagittal" and _d3_side(files[0])<0
            i0=int(round(D3_BAND[0]*(len(files)-1))); i1=int(round(D3_BAND[1]*(len(files)-1))); avail=list(range(i0,i1+1))
            if len(avail)>=D3_NSLICE: picks=[avail[int(round(x))] for x in np.linspace(0,len(avail)-1,D3_NSLICE)]; off=0
            else: picks=avail; off=(D3_NSLICE-len(picks))//2
            for c,p in enumerate(picks):
                img=_d3_render(files[p],flip)
                if img is None: img=_d3_render(files[min(len(files)-1,p+1)],flip)
                if img is not None: out[slot,off+c]=(img*255).astype(np.uint8)
            mask[slot]=len(picks)
    return idx,out,mask


def _d3_models():
    import timm
    root=_d3_root(); models=[]; metrics=[]; folds=[]; cfg_ref=None
    for path in sorted(root.glob("*_f*.pt")):
        z=torch.load(path,map_location="cpu",weights_only=False); cfg=z["cfg"]; stem=cfg.get("stem","native"); in_ch=3 if stem=="compress" else int(cfg.get("n_slice",16)); enc=timm.create_model(cfg["backbone"],pretrained=False,num_classes=0,in_chans=in_ch,**({"img_size":cfg["img"]} if "vit_" in cfg["backbone"] else {}));
        if cfg.get("cond")=="token": enc=D3SlotToken(enc,6)
        model=D3Net(enc,cfg.get("cond"),int(cfg.get("n_meta",0)),cfg.get("pool","xcodex"),stem=stem,n_slice=int(cfg.get("n_slice",16)))
        missing,unexpected=model.load_state_dict(z["state_dict"],strict=False)
        if [k for k in missing if not k.startswith("enc.")] or unexpected: raise RuntimeError(f"DINOv3 state mismatch: {missing[:3]} {unexpected[:3]}")
        models.append(model.to(DEVICE).eval()); metrics.append(checkpoint_metric_vector(z)); folds.append(int(z.get("fold",len(folds)))); cfg_ref=cfg
    return root,models,metrics,folds,cfg_ref


def _d3_norm(im,cfg):
    kind=cfg.get("norm","none")
    if kind=="zscore":
        m=(im>0).float(); n=m.sum((1,2,3),keepdim=True).clamp_min(1); mu=(im*m).sum((1,2,3),keepdim=True)/n; var=(((im-mu)*m)**2).sum((1,2,3),keepdim=True)/n; return (im-mu)/(var.sqrt()+1e-6)*m
    if kind=="imagenet": return (im-0.485)/0.229*(im>0)
    return im

@torch.inference_mode()
def _d3_predict_block(models,cfg,images,masks):
    ims=[]; slots=[]; sidx=[]; vms=[]
    for b in range(len(masks)):
        present=np.flatnonzero(masks[b]>0)
        if not len(present): continue
        blk=images[b][present]; ims.append(torch.from_numpy(blk)); vms.append(torch.from_numpy(blk.reshape(len(blk),blk.shape[1],-1).max(2)>0)); slots.append(torch.from_numpy(present+1).long()); sidx.append(torch.full((len(present),),b,dtype=torch.long))
    out=np.full((len(models),len(masks),len(LABELS)),np.nan,np.float32)
    if not ims: return out
    im=_d3_norm(torch.cat(ims).to(DEVICE,non_blocking=True).float().div_(255),cfg); sl=torch.cat(slots).to(DEVICE); si=torch.cat(sidx).to(DEVICE); vm=torch.cat(vms).to(DEVICE); sm=torch.zeros(len(sl),int(cfg.get("n_meta",0)),device=DEVICE)
    dtype=torch.float16
    with torch.autocast("cuda",dtype=dtype):
        for mi,m in enumerate(models): out[mi]=torch.sigmoid(m(im,sl,sm,si,len(masks),vm=vm).float()).cpu().numpy()
    return out


def run_dino3() -> dict:
    root,models,metrics,folds,cfg=_d3_models(); by=_d3_series_data(); preds=np.full((len(models),len(TEST_IDS),len(LABELS)),np.nan,np.float32); workers=max(1,min(4,os.cpu_count() or 4)); done=0
    with ProcessPoolExecutor(max_workers=workers) as ex:
        for start in range(0,len(TEST_IDS),SPEC.dino3_chunk):
            block=TEST_IDS[start:start+SPEC.dino3_chunk]; imgs=np.zeros((len(block),6,D3_NSLICE,D3_SIZE,D3_SIZE),np.uint8); masks=np.zeros((len(block),6),np.uint8); fut=[ex.submit(_d3_build_study,(i,u,by.get(u,[]))) for i,u in enumerate(block)]
            for f in as_completed(fut):
                i,a,m=f.result(); imgs[i]=a; masks[i]=m
            for a in range(0,len(block),SPEC.dino3_micro):
                b=min(len(block),a+SPEC.dino3_micro); preds[:,start+a:start+b]=_d3_predict_block(models,cfg,imgs[a:b],masks[a:b])
            done += len(block); stamp(f"DINOv3 {done}/{len(TEST_IDS)}")
            del imgs,masks; gc.collect()
    rank_bank = np.stack([rank_columns(p) for p in preds])
    metric_matrix = None if any(x is None for x in metrics) else np.stack(metrics)
    weights = stable_softmax_weights(metric_matrix, len(models))
    equal = rank_bank.mean(axis=0)
    weighted = weighted_rank_bank(rank_bank, weights)
    robust = robust_rank_consensus(rank_bank)
    consistent, consistency_weights = consistency_weighted_rank(
        rank_bank, strength=0.32
    )
    probmean = rank_columns(preds.mean(axis=0))
    enhanced = (
        0.62 * equal
        + 0.18 * consistent
        + 0.12 * robust
        + 0.08 * weighted
    )
    result = {
        "equal": equal,
        "weighted": weighted,
        "robust": robust,
        "consistent": consistent,
        "consistency_weights": consistency_weights,
        "enhanced": enhanced,
        "probmean": probmean,
        "bank": rank_bank,
        "metrics": metric_matrix,
        "folds": folds,
        "root": root,
    }
    del models
    release_cuda()
    return result


In [ ]:
# -------------------------- Dual-family RadImageNet --------------------------
from torchvision.models import resnet50

RAD_PLANES = ("Sagittal", "Coronal", "Axial")
RAD_SLICES = 8
RAD_SIZE = 224
RAD_BAND = (0.12, 0.88)
RAD_TOKEN = 2048
RAD_HIDDEN = 512

RAD_ENCODER_SHA = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
RAD_PUBLIC_SHA = "0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa"
RAD_NATIVE_FOLD_SHA = "1301603a060226c47c96be54d4c3618fee41f2e97f8f82d8f77a752819ffb7e3"
RAD_NATIVE_CONFIG = "794a44d95a0ebb7096f17daa5a06dc191ec16c4d0835e69ac1868e82a7eb05dd"
RAD_NATIVE_HEAD_SHA = {
    "rad_head_f0.pt": "0c92b27578e139cc35071a3f72ddd4e1a66106761225a7e011f076f37eb7051d",
    "rad_head_f1.pt": "c29e60a99982d8dfb933a276f51c8b3a7d2e849649ed27a09affa822405fbd17",
    "rad_head_f2.pt": "40328ac7d72ca281e7e04438643100e99eb87fbe1e2b51965350b1a73a7b57ab",
    "rad_head_f3.pt": "6538a5faa92b61705727a14bd98c5ddce2989028dd8cccb261c47b2066fa5efa",
    "rad_head_f4.pt": "5bd126d68ddadd479a0eab6f0cdb17603060614a7eb14402e5b9945f651107af",
}


class RadFeatureEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            *list(resnet50(weights=None).children())[:-2]
        )

    def forward(self, image):
        return self.features(image).mean(dim=(2, 3))


class RadAttentionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.project = nn.Sequential(
            nn.LayerNorm(RAD_TOKEN),
            nn.Linear(RAD_TOKEN, RAD_HIDDEN),
            nn.GELU(),
        )
        self.plane_embedding = nn.Parameter(
            torch.randn(3, RAD_HIDDEN) * 0.01
        )
        self.position_embedding = nn.Parameter(
            torch.randn(RAD_SLICES, RAD_HIDDEN) * 0.01
        )
        self.target_query = nn.Parameter(
            torch.randn(len(LABELS), RAD_HIDDEN) * 0.02
        )
        self.attention = nn.MultiheadAttention(
            RAD_HIDDEN, 8, dropout=0.10, batch_first=True
        )
        self.fusion = nn.Sequential(
            nn.LayerNorm(RAD_HIDDEN * 4),
            nn.Linear(RAD_HIDDEN * 4, RAD_HIDDEN),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        self.target_weight = nn.Parameter(
            torch.randn(len(LABELS), RAD_HIDDEN) * 0.02
        )
        self.target_bias = nn.Parameter(torch.zeros(len(LABELS)))

    def forward(self, feature, token_mask):
        token = self.project(feature.float()).view(
            len(feature), 3, RAD_SLICES, RAD_HIDDEN
        )
        token = (
            token
            + self.plane_embedding[None, :, None]
            + self.position_embedding[None, None]
        )
        token = token.flatten(1, 2)

        padding = token_mask <= 0
        empty = padding.all(dim=1)
        if empty.any():
            padding = padding.clone()
            padding[empty, 0] = False

        query = self.target_query[None].expand(len(token), -1, -1)
        attended = query + self.attention(
            query,
            token,
            token,
            key_padding_mask=padding,
            need_weights=False,
        )[0]

        denominator = (
            token_mask.sum(dim=1, keepdim=True)
            .clamp_min(1)
            .unsqueeze(-1)
        )
        mean = (
            (token * token_mask.unsqueeze(-1)).sum(dim=1, keepdim=True)
            / denominator
        )
        mean = mean.expand(-1, len(LABELS), -1)

        fused = self.fusion(torch.cat(
            [
                attended,
                mean,
                (attended - mean).abs(),
                attended * mean,
            ],
            dim=-1,
        ))
        return (
            fused * self.target_weight[None]
        ).sum(dim=-1) + self.target_bias


def _rad_adapt_encoder_state(state_dict, model):
    adapted = {}
    for key, value in state_dict.items():
        if key.startswith("backbone."):
            new_key = "features." + key[len("backbone."):]
        else:
            new_key = key
        adapted[new_key] = value

    expected = set(model.state_dict().keys())
    observed = set(adapted.keys())
    missing = sorted(expected - observed)
    unexpected = sorted(observed - expected)
    if missing or unexpected:
        raise RuntimeError(
            "Rad encoder checkpoint contract mismatch after adaptation: "
            f"missing={missing[:8]} unexpected={unexpected[:8]}"
        )
    return adapted


def _rad_adapt_head_state(state_dict, model):
    prefix_map = (
        ("plane", "plane_embedding"),
        ("position", "position_embedding"),
        ("query", "target_query"),
        ("attn.", "attention."),
        ("fuse.", "fusion."),
        ("weight", "target_weight"),
        ("bias", "target_bias"),
    )

    adapted = {}
    for key, value in state_dict.items():
        new_key = key

        if key == "plane":
            new_key = "plane_embedding"
        elif key == "position":
            new_key = "position_embedding"
        elif key == "query":
            new_key = "target_query"
        elif key.startswith("attn."):
            new_key = "attention." + key[len("attn."):]
        elif key.startswith("fuse."):
            new_key = "fusion." + key[len("fuse."):]
        elif key == "weight":
            new_key = "target_weight"
        elif key == "bias":
            new_key = "target_bias"

        if new_key in adapted:
            raise RuntimeError(
                f"duplicate Rad head key after adaptation: {new_key}"
            )
        adapted[new_key] = value

    expected = set(model.state_dict().keys())
    observed = set(adapted.keys())
    missing = sorted(expected - observed)
    unexpected = sorted(observed - expected)
    if missing or unexpected:
        raise RuntimeError(
            "Rad head checkpoint contract mismatch after adaptation: "
            f"missing={missing[:8]} unexpected={unexpected[:8]}"
        )
    return adapted


def _rad_find_sha(name: str, expected_sha: str) -> Path:
    for path in Path("/kaggle/input").glob(f"**/{name}"):
        try:
            if sha256(path) == expected_sha:
                return path
        except Exception:
            continue
    raise FileNotFoundError(f"RadImageNet asset {name} was not found")


def _rad_native_dir() -> tuple[Path, dict]:
    for path in Path("/kaggle/input").glob("**/rad_heads_manifest.json"):
        try:
            manifest = json.loads(path.read_text())
        except Exception:
            continue
        if manifest.get("artifact_type") != "rsna-radimagenet-foldsv1-heads":
            continue
        if manifest.get("fold_sha256") != RAD_NATIVE_FOLD_SHA:
            continue
        if manifest.get("config_hash") != RAD_NATIVE_CONFIG:
            continue
        if manifest.get("labels") != LABELS:
            continue
        return path.parent, manifest
    raise FileNotFoundError("Native RadImageNet five-fold heads were not found")


def _rad_fatsat(ds) -> bool:
    if ds is None:
        return False
    description = _MULTI_SEP.sub(
        " ",
        f"{getattr(ds, 'SeriesDescription', '') or ''} "
        f"{getattr(ds, 'SequenceName', '') or ''}".lower(),
    )
    options = getattr(ds, "ScanOptions", None)
    try:
        items = (
            list(options)
            if not isinstance(options, str)
            else re.split(r"[|\\]", options)
        )
    except Exception:
        items = []
    return bool(
        _FAT_RX.search(description)
        or any(
            str(item).strip().upper() in _FAT_OPTIONS
            for item in items
        )
    )


def _rad_series_table() -> pd.DataFrame:
    series = pd.read_csv(
        COMP / "test_series.csv",
        dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
    )
    root = COMP / "test_series"

    def inspect(row):
        uid, sid, plane = row
        folder = root / uid / sid
        files = dicom_files(folder)
        ds = header(files[len(files) // 2]) if files else None
        tag = str(getattr(ds, "Laterality", "") or "").strip().upper()[:1]
        if tag not in ("L", "R"):
            tag = str(
                getattr(ds, "ImageLaterality", "") or ""
            ).strip().upper()[:1]
        if tag not in ("L", "R"):
            tag = None
        corner = None
        try:
            corner = float(ds.ImagePositionPatient[0])
        except Exception:
            pass
        return {
            "uid": uid,
            "sid": sid,
            "plane": plane,
            "folder": folder,
            "files": files,
            "count": len(files),
            "fs": _rad_fatsat(ds),
            "laterality": tag,
            "center_x": image_center_x(ds),
            "corner_x": corner,
        }

    jobs = [
        (
            str(row.StudyInstanceUID),
            str(row.SeriesInstanceUID),
            str(row.Anatomical_Plane),
        )
        for row in series.itertuples(index=False)
        if str(row.StudyInstanceUID) in set(TEST_IDS)
    ]
    with ThreadPoolExecutor(
        max_workers=min(16, os.cpu_count() or 4)
    ) as pool:
        return pd.DataFrame(list(pool.map(inspect, jobs)))


def _rad_select(table: pd.DataFrame, legacy_side: bool):
    selected = {uid: [None, None, None] for uid in TEST_IDS}
    sides = {}

    for uid, group in table.groupby("uid"):
        for plane_index, plane in enumerate(RAD_PLANES):
            candidates = group[
                (group["plane"] == plane) & (group["fs"])
            ]
            if len(candidates):
                row = candidates.sort_values(
                    "count", ascending=False
                ).iloc[0]
                selected[str(uid)][plane_index] = row.to_dict()

        tags = [
            item for item in group["laterality"].tolist()
            if item in ("L", "R")
        ]
        if tags:
            sides[str(uid)] = tags[0]
            continue

        key = "corner_x" if legacy_side else "center_x"
        values = [
            float(item)
            for item in group[key].tolist()
            if item is not None and np.isfinite(item)
        ]
        if not values:
            sides[str(uid)] = None
            continue
        location = float(np.median(values))
        dead_zone = 5.0 if legacy_side else 20.0
        sides[str(uid)] = (
            None
            if abs(location) < dead_zone
            else ("R" if location < 0 else "L")
        )

    return selected, sides


def _rad_public_order(record: dict) -> list[Path]:
    return physical_order(
        Path(record["folder"]),
        fallback_dominant=True,
    )


def _rad_native_order(record: dict) -> list[Path]:
    files = dicom_files(Path(record["folder"]))
    projected = []
    for position, path in enumerate(files):
        ds = header(
            path,
            ["ImagePositionPatient", "ImageOrientationPatient"],
        )
        key = None
        if ds is not None:
            ipp = vector(
                getattr(ds, "ImagePositionPatient", None), 3
            )
            iop = vector(
                getattr(ds, "ImageOrientationPatient", None), 6
            )
            if ipp is not None and iop is not None:
                key = float(
                    np.dot(ipp, np.cross(iop[:3], iop[3:]))
                )
        projected.append((key, position, path))

    # Native heads were trained with a single ordering system per series.
    if any(key is None for key, _, _ in projected):
        return files
    return [
        path
        for key, _, path in sorted(projected, key=lambda row: row[0])
    ]


def _rad_volume(record: dict, legacy: bool) -> np.ndarray | None:
    files = (
        _rad_public_order(record)
        if legacy
        else _rad_native_order(record)
    )
    if not files:
        return None

    lo = int(RAD_BAND[0] * (len(files) - 1))
    hi = int(RAD_BAND[1] * (len(files) - 1))
    indices = (
        np.unique(np.linspace(lo, hi, RAD_SLICES).astype(np.int64))
        if hi > lo
        else np.asarray([len(files) // 2], np.int64)
    )
    while len(indices) < RAD_SLICES:
        indices = np.append(indices, indices[-1])
    indices = indices[:RAD_SLICES]

    planes = [decode_plane(files[int(index)])[0] for index in indices]
    valid = [i for i, image in enumerate(planes) if image is not None]
    if not valid:
        return None

    shape = planes[valid[0]].shape
    if legacy:
        # Public v15 used zero filling for a slice that failed to decode.
        planes = [
            np.zeros(shape, np.float32) if image is None else image
            for image in planes
        ]
    else:
        # Native folds used the nearest decodable acquired slice.
        for i, image in enumerate(planes):
            if image is None:
                nearest = min(valid, key=lambda j: abs(j - i))
                planes[i] = planes[nearest]

    planes = [
        image if image.shape == shape else np.zeros(shape, np.float32)
        for image in planes
    ]
    volume = percentile_scale(np.stack(planes))
    return resize_stack(volume, RAD_SIZE)


def _rad_build_study(job, legacy: bool):
    index, uid, records, side = job
    pixels = np.zeros(
        (3, RAD_SLICES, RAD_SIZE, RAD_SIZE),
        np.uint8,
    )
    slot_mask = np.zeros(3, np.uint8)

    for slot, record in enumerate(records):
        if record is None:
            continue
        volume = _rad_volume(record, legacy=legacy)
        if volume is None:
            continue

        if side == "R":
            if record["plane"] == "Sagittal":
                volume = volume[::-1].copy()
            else:
                volume = volume[:, :, ::-1].copy()

        pixels[slot] = volume
        slot_mask[slot] = 1

    return index, pixels, slot_mask


def _rad_encoder() -> RadFeatureEncoder:
    path = _rad_find_sha("ResNet50.pt", RAD_ENCODER_SHA)
    model = RadFeatureEncoder()
    raw_state = torch.load(
        path, map_location="cpu", weights_only=True
    )
    model.load_state_dict(
        _rad_adapt_encoder_state(raw_state, model),
        strict=True,
    )
    del raw_state
    if sum(p.numel() for p in model.parameters()) != 23_508_032:
        raise RuntimeError("RadImageNet encoder parameter count drift")
    model.eval().to(DEVICE)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def _rad_public_heads():
    path = _rad_find_sha("v52_radimagenet_heads.pt", RAD_PUBLIC_SHA)
    payload = torch.load(path, map_location="cpu", weights_only=True)

    expected = {
        "version": "v52-radimagenet-resnet50-official-1",
        "targets": LABELS,
        "encoder_sha256": RAD_ENCODER_SHA,
        "img": 224,
        "slices_per_plane": 8,
        "feature": "global_average_pool",
    }
    for key, value in expected.items():
        if payload.get(key) != value:
            raise RuntimeError(f"public RadImageNet contract drift: {key}")

    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError("public RadImageNet requires five folds")

    heads, metrics = [], []
    for record in sorted(folds, key=lambda item: int(item.get("fold", -1))):
        head = RadAttentionHead().to(DEVICE).eval()
        head.load_state_dict(
            _rad_adapt_head_state(record["state_dict"], head),
            strict=True,
        )
        heads.append(head)
        metrics.append(checkpoint_metric_vector(record))
    return heads, metrics


def _rad_native_heads():
    root, manifest = _rad_native_dir()
    heads, metrics = [], []

    for fold in range(5):
        name = f"rad_head_f{fold}.pt"
        path = root / name
        if sha256(path) != RAD_NATIVE_HEAD_SHA[name]:
            raise RuntimeError(f"native RadImageNet SHA drift: {name}")
        if manifest["heads"][name]["sha256"] != RAD_NATIVE_HEAD_SHA[name]:
            raise RuntimeError(f"native RadImageNet manifest drift: {name}")

        checkpoint = torch.load(
            path, map_location="cpu", weights_only=False
        )
        if checkpoint.get("fold") != fold:
            raise RuntimeError(f"native RadImageNet fold drift: {fold}")
        if checkpoint.get("config_hash") != RAD_NATIVE_CONFIG:
            raise RuntimeError(f"native RadImageNet config drift: {fold}")

        head = RadAttentionHead().to(DEVICE).eval()
        head.load_state_dict(
            _rad_adapt_head_state(checkpoint["state_dict"], head),
            strict=True,
        )
        heads.append(head)
        metrics.append(checkpoint_metric_vector(checkpoint))

    return heads, metrics


@torch.inference_mode()
def _rad_encode(encoder, pixels, slot_mask):
    n = len(pixels)
    features = np.zeros(
        (n, 3 * RAD_SLICES, RAD_TOKEN),
        np.float16,
    )
    token_mask = np.repeat(
        slot_mask[:, :, None],
        RAD_SLICES,
        axis=2,
    ).reshape(n, -1)
    valid = np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = pixels.reshape(-1, RAD_SIZE, RAD_SIZE)

    batch_size = 96
    for start in range(0, len(valid), batch_size):
        indices = valid[start:start + batch_size]
        image = (
            torch.from_numpy(flat[indices])
            .to(DEVICE)
            .float()
            .div_(127.5)
            .sub_(1.0)
            .unsqueeze(1)
            .expand(-1, 3, -1, -1)
            .contiguous()
        )
        with torch.autocast("cuda", dtype=torch.float16):
            feature = encoder(image)
        values = feature.float().cpu().numpy()
        if not np.isfinite(values).all():
            raise RuntimeError("non-finite RadImageNet feature")
        features.reshape(-1, RAD_TOKEN)[indices] = values.astype(
            np.float16
        )
        del image, feature

    return features, token_mask.astype(np.float32)


@torch.inference_mode()
def _rad_predict_heads(heads, features, masks, use_amp: bool):
    feature = torch.from_numpy(features).to(DEVICE)
    mask = torch.from_numpy(masks).to(DEVICE)
    output = []

    for head in heads:
        if use_amp:
            with torch.autocast("cuda", dtype=torch.float16):
                probability = torch.sigmoid(head(feature, mask)).float()
        else:
            probability = torch.sigmoid(head(feature, mask)).float()
        output.append(probability.cpu().numpy())

    del feature, mask
    return output


def _rad_family_predictions(
    encoder,
    heads,
    selected,
    sides,
    legacy: bool,
    use_head_amp: bool,
):
    predictions = [
        np.full((len(TEST_IDS), len(LABELS)), np.nan, np.float32)
        for _ in heads
    ]
    workers = max(1, min(4, os.cpu_count() or 4))

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for start in range(0, len(TEST_IDS), SPEC.rad_block):
            block = TEST_IDS[start:start + SPEC.rad_block]
            pixels = np.zeros(
                (len(block), 3, RAD_SLICES, RAD_SIZE, RAD_SIZE),
                np.uint8,
            )
            masks = np.zeros((len(block), 3), np.uint8)

            jobs = [
                (i, uid, selected[uid], sides.get(uid))
                for i, uid in enumerate(block)
            ]
            for index, study_pixels, study_mask in pool.map(
                lambda job: _rad_build_study(job, legacy=legacy),
                jobs,
            ):
                pixels[index] = study_pixels
                masks[index] = study_mask

            features, token_masks = _rad_encode(
                encoder, pixels, masks
            )
            block_predictions = _rad_predict_heads(
                heads,
                features,
                token_masks,
                use_amp=use_head_amp,
            )
            for fold, values in enumerate(block_predictions):
                predictions[fold][
                    start:start + len(block)
                ] = values

            stamp(
                f"RadImageNet {'public' if legacy else 'native'} "
                f"{start + len(block)}/{len(TEST_IDS)}"
            )
            del pixels, masks, features, token_masks, block_predictions
            gc.collect()

    if not all(np.isfinite(item).all() for item in predictions):
        raise RuntimeError("RadImageNet predictions are incomplete")
    return np.stack(predictions)


def _rad_metric_matrix(metrics):
    return None if any(item is None for item in metrics) else np.stack(metrics)


def run_rad_dual() -> dict:
    table = _rad_series_table()
    public_selected, public_sides = _rad_select(
        table, legacy_side=True
    )
    native_selected, native_sides = _rad_select(
        table, legacy_side=False
    )

    encoder = _rad_encoder()

    public_heads, public_metrics_list = _rad_public_heads()
    public_prob = _rad_family_predictions(
        encoder,
        public_heads,
        public_selected,
        public_sides,
        legacy=True,
        use_head_amp=True,
    )
    del public_heads
    release_cuda()

    native_heads, native_metrics_list = _rad_native_heads()
    native_prob = _rad_family_predictions(
        encoder,
        native_heads,
        native_selected,
        native_sides,
        legacy=False,
        use_head_amp=False,
    )
    del native_heads, encoder
    release_cuda()

    public_bank = np.stack([
        rank_columns(item) for item in public_prob
    ])
    native_bank = np.stack([
        rank_columns(item) for item in native_prob
    ])

    public_probability_rank = rank_columns(public_prob.mean(axis=0))
    public_fold_rank = public_bank.mean(axis=0)
    native_fold_rank = native_bank.mean(axis=0)

    public_robust = robust_rank_consensus(public_bank)
    native_robust = robust_rank_consensus(native_bank)
    public_consistent, public_weights = consistency_weighted_rank(
        public_bank, strength=0.28
    )
    native_consistent, native_weights = consistency_weighted_rank(
        native_bank, strength=0.28
    )

    public_metrics = _rad_metric_matrix(public_metrics_list)
    native_metrics = _rad_metric_matrix(native_metrics_list)
    public_metric_rank = weighted_rank_bank(
        public_bank,
        stable_softmax_weights(public_metrics, 5),
    )
    native_metric_rank = weighted_rank_bank(
        native_bank,
        stable_softmax_weights(native_metrics, 5),
    )

    # Keep the public probability-mean estimator dominant because that is the
    # verified public-v15 contract. Native keeps fold-rank mean dominant.
    public_enhanced = (
        0.72 * public_probability_rank
        + 0.12 * public_fold_rank
        + 0.10 * public_consistent
        + 0.04 * public_robust
        + 0.02 * public_metric_rank
    )
    native_enhanced = (
        0.72 * native_fold_rank
        + 0.12 * native_consistent
        + 0.10 * native_robust
        + 0.06 * native_metric_rank
    )

    anchor = 0.50 * public_probability_rank + 0.50 * native_fold_rank

    return {
        "anchor": anchor,
        "public_anchor": public_probability_rank,
        "native_anchor": native_fold_rank,
        "public_enhanced": public_enhanced,
        "native_enhanced": native_enhanced,
        "public_bank": public_bank,
        "native_bank": native_bank,
        "public_metrics": public_metrics,
        "native_metrics": native_metrics,
        "public_consistency_weights": public_weights,
        "native_consistency_weights": native_weights,
    }


In [ ]:
ROOT = COMP  # V24 fix: E11 alpha chooser referenced undefined ROOT

# ----------------------- Diverse Rad specialist -----------------------

class DiverseRadHead(nn.Module):
    """Independent four-slot RadImageNet attention head."""

    def __init__(self, n_slot: int = 4, n_slice: int = 8):
        super().__init__()
        self.n_slot = int(n_slot)
        self.n_slice = int(n_slice)

        self.project = nn.Sequential(
            nn.LayerNorm(2048),
            nn.Linear(2048, 512),
            nn.GELU(),
        )
        self.plane = nn.Parameter(
            torch.randn(self.n_slot, 512) * 0.01
        )
        self.position = nn.Parameter(
            torch.randn(self.n_slice, 512) * 0.01
        )
        self.query = nn.Parameter(
            torch.randn(len(LABELS), 512) * 0.02
        )
        self.attn = nn.MultiheadAttention(
            512, 8, dropout=0.10, batch_first=True
        )
        self.fuse = nn.Sequential(
            nn.LayerNorm(2048),
            nn.Linear(2048, 512),
            nn.GELU(),
            nn.Dropout(0.15),
        )
        self.weight = nn.Parameter(
            torch.randn(len(LABELS), 512) * 0.02
        )
        self.bias = nn.Parameter(
            torch.zeros(len(LABELS))
        )

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(
            len(token),
            self.n_slot,
            self.n_slice,
            512,
        )
        token = (
            token
            + self.plane[None, :, None]
            + self.position[None, None]
        )
        token = token.flatten(1, 2)

        padding = mask <= 0
        empty = padding.all(dim=1)
        if empty.any():
            padding = padding.clone()
            padding[empty, 0] = False

        query = self.query[None].expand(
            len(token), -1, -1
        )
        attended = query + self.attn(
            query,
            token,
            token,
            key_padding_mask=padding,
            need_weights=False,
        )[0]

        denom = (
            mask.sum(1, keepdim=True)
            .clamp_min(1)
            .unsqueeze(-1)
        )
        mean = (
            (token * mask.unsqueeze(-1)).sum(1, keepdim=True)
            / denom
        )
        mean = mean.expand(
            -1,
            len(LABELS),
            -1,
        )

        fused = self.fuse(torch.cat(
            [
                attended,
                mean,
                (attended - mean).abs(),
                attended * mean,
            ],
            dim=-1,
        ))
        return (
            fused * self.weight[None]
        ).sum(-1) + self.bias


E11_SLOTS_LOCAL = (
    ("SAG_NOFS", "Sagittal", False),
    ("COR_NOFS", "Coronal", False),
    ("AX_NOFS", "Axial", False),
    ("SAG_FS", "Sagittal", True),
)
E11_CROP_MM_LOCAL = 130.0
E11_SLICES_LOCAL = 8
E11_SIZE_LOCAL = 224


def _e11_slot_records(table):
    result = {
        str(uid): {
            name: None for name, _, _ in E11_SLOTS_LOCAL
        }
        for uid in TEST_IDS
    }

    for uid, group in table.groupby("uid"):
        uid = str(uid)
        for name, plane, fs in E11_SLOTS_LOCAL:
            candidates = group[
                (group["plane"] == plane)
                & (group["fs"] == bool(fs))
            ]
            if len(candidates):
                row = candidates.sort_values(
                    ["count", "center_x"],
                    ascending=[False, True],
                ).iloc[0]
                result[uid][name] = row.to_dict()

    return result


def _e11_side(table):
    sides = {}
    for uid, group in table.groupby("uid"):
        tags = [
            str(x).strip().upper()[:1]
            for x in group["laterality"].tolist()
            if str(x).strip().upper()[:1] in ("L", "R")
        ]
        if tags:
            sides[str(uid)] = tags[0]
            continue

        values = [
            float(x)
            for x in group["center_x"].tolist()
            if x is not None and np.isfinite(x)
        ]
        if values:
            med = float(np.median(values))
            sides[str(uid)] = (
                None
                if abs(med) < 20.0
                else ("R" if med < 0 else "L")
            )
        else:
            sides[str(uid)] = None
    return sides


def _e11_read_series(record):
    files = _rad_native_order(record)
    if not files:
        return None

    n = len(files)
    lo = int(0.12 * (n - 1))
    hi = int(0.88 * (n - 1))
    indices = (
        np.unique(
            np.linspace(
                lo,
                hi,
                E11_SLICES_LOCAL,
            ).astype(int)
        )
        if hi > lo
        else np.asarray([n // 2], dtype=int)
    )
    while len(indices) < E11_SLICES_LOCAL:
        indices = np.append(indices, indices[-1])
    indices = indices[:E11_SLICES_LOCAL]

    planes = []
    spacing = None

    for index in indices:
        path = files[int(index)]
        array, ds = decode_plane(path)
        if spacing is None and ds is not None:
            try:
                spacing = float(ds.PixelSpacing[0])
            except Exception:
                spacing = None
        planes.append(array)

    valid = [i for i,p in enumerate(planes) if p is not None]
    if not valid:
        return None

    for index, image in enumerate(planes):
        if image is None:
            nearest = min(
                valid,
                key=lambda j: abs(j - index),
            )
            planes[index] = planes[nearest]

    shape = planes[0].shape
    planes = [
        image
        if image is not None and image.shape == shape
        else np.zeros(shape, np.float32)
        for image in planes
    ]

    volume = np.stack(planes)

    if spacing is not None and np.isfinite(spacing) and spacing > 0:
        width = int(round(E11_CROP_MM_LOCAL / spacing))
        height, width_image = shape
        if 16 < width < min(height, width_image):
            cy = height // 2
            cx = width_image // 2
            half = width // 2
            volume = volume[
                :,
                max(0, cy-half):cy+half,
                max(0, cx-half):cx+half,
            ]

    low, high = np.percentile(volume, [1, 99])
    volume = np.clip(
        (volume - low)
        / max(float(high-low), 1e-6),
        0.0,
        1.0,
    )

    tensor = torch.from_numpy(
        np.ascontiguousarray(volume)
    ).unsqueeze(0)
    tensor = F.interpolate(
        tensor,
        size=(E11_SIZE_LOCAL, E11_SIZE_LOCAL),
        mode="bilinear",
        align_corners=False,
    )
    return (
        tensor.squeeze(0) * 255.0
    ).round().clamp(0, 255).to(torch.uint8).numpy()


def _e11_build_cache(table):
    selected = _e11_slot_records(table)
    side = _e11_side(table)

    studies = list(TEST_IDS)
    cache = np.zeros(
        (
            len(studies),
            len(E11_SLOTS_LOCAL),
            E11_SLICES_LOCAL,
            E11_SIZE_LOCAL,
            E11_SIZE_LOCAL,
        ),
        np.uint8,
    )
    mask = np.zeros(
        (
            len(studies),
            len(E11_SLOTS_LOCAL),
        ),
        np.float32,
    )

    for i, uid in enumerate(studies):
        for k, (name, _, _) in enumerate(E11_SLOTS_LOCAL):
            record = selected.get(uid, {}).get(name)
            if record is None:
                continue

            volume = _e11_read_series(record)
            if volume is None:
                continue

            if side.get(uid) == "R":
                if record["plane"] == "Sagittal":
                    volume = volume[::-1].copy()
                else:
                    volume = volume[:, :, ::-1].copy()

            cache[i, k] = volume
            mask[i, k] = 1.0

    return cache, mask


@torch.inference_mode()
def _e11_encode(
    encoder,
    pixels,
    slot_mask,
    device,
):
    n = len(pixels)
    token_count = (
        len(E11_SLOTS_LOCAL)
        * E11_SLICES_LOCAL
    )
    features = np.zeros(
        (
            n,
            token_count,
            2048,
        ),
        np.float16,
    )
    masks = np.repeat(
        slot_mask[:, :, None],
        E11_SLICES_LOCAL,
        axis=2,
    ).reshape(n, token_count)

    valid = np.flatnonzero(
        masks.reshape(-1) > 0
    )
    flat = pixels.reshape(
        -1,
        E11_SIZE_LOCAL,
        E11_SIZE_LOCAL,
    )

    for start in range(0, len(valid), 96):
        ids = valid[start:start + 96]
        images = (
            torch.from_numpy(
                flat[ids]
            )
            .to(device)
            .float()
            .div_(127.5)
            .sub_(1.0)
            .unsqueeze(1)
            .expand(-1, 3, -1, -1)
            .contiguous()
        )

        with torch.autocast(
            "cuda",
            dtype=torch.float16,
        ):
            encoded = encoder(images)

        features.reshape(-1, 2048)[ids] = (
            encoded.float()
            .cpu()
            .numpy()
            .astype(np.float16)
        )

        del images, encoded

    return features, masks.astype(np.float32)



def _e11_find_artifact(filename: str) -> Path:
    """Resolve an attached Kaggle artifact without external helper dependencies."""
    roots = [Path("/kaggle/input"), Path("/kaggle/working")]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        matches.extend(p for p in root.rglob(filename) if p.is_file())

    unique = {}
    for p in matches:
        unique[str(p.resolve())] = p

    if not unique:
        raise FileNotFoundError(
            f"Attached artifact not found: {filename}"
        )

    return sorted(
        unique.values(),
        key=lambda p: (len(p.parts), str(p))
    )[0]


def _e11_load_heads():
    path = _e11_find_artifact("v52_e11_heads.pt")
    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError(
            "E11 head package must contain five folds"
        )

    heads = []
    for record in sorted(
        folds,
        key=lambda r: int(r.get("fold", -1)),
    ):
        model = DiverseRadHead(
            len(E11_SLOTS_LOCAL),
            E11_SLICES_LOCAL,
        ).to(DEVICE).eval()
        model.load_state_dict(
            record["state_dict"],
            strict=True,
        )
        heads.append(model)

    return heads, payload


@torch.inference_mode()
def _e11_predict():
    table = _rad_series_table()
    pixels, slot_mask = _e11_build_cache(table)
    encoder = _rad_encoder()

    features, masks = _e11_encode(
        encoder,
        pixels,
        slot_mask,
        DEVICE,
    )

    del encoder, pixels, slot_mask
    release_cuda()

    heads, payload = _e11_load_heads()

    predictions = []
    for head in heads:
        predictions.append(
            torch.sigmoid(
                head(
                    torch.from_numpy(features).to(DEVICE),
                    torch.from_numpy(masks).to(DEVICE),
                )
            )
            .float()
            .cpu()
            .numpy()
        )
        del head
        release_cuda()

    del features, masks
    release_cuda()

    bank = np.stack(predictions)
    return {
        "probability": bank.mean(axis=0),
        "rank_bank": np.stack(
            [rank_columns(x) for x in bank]
        ),
        "heads": len(heads),
        "payload": payload,
    }


def _e11_choose_target_alpha():
    """
    Pick a small target-specific residual from already-available OOF predictions.

    The selection is deliberately conservative. A target can move away from zero only
    when the average grouped held-out AUC gain is positive; otherwise it keeps the
    proven alpha=0 fallback.
    """
    try:
        e11_oof_path = _e11_find_artifact("v52_e11_oof.csv")
        base_oof_path = _e11_find_artifact("oof.npz")
    except FileNotFoundError:
        return {
            target: 0.20
            for target in LABELS
        }

    e11_frame = pd.read_csv(
        e11_oof_path,
        dtype={"StudyInstanceUID": str},
    )
    train = pd.read_csv(
        ROOT / "train.csv",
        dtype={"StudyInstanceUID": str},
    )

    with np.load(
        base_oof_path,
        allow_pickle=False,
    ) as bundle:
        ids = bundle["ids"].astype(str)
        base_prob = bundle["pred"].astype(float)
        gold = bundle["gold_mask"].astype(bool)

    aligned_e11 = (
        train[["StudyInstanceUID"]]
        .merge(
            e11_frame[
                ["StudyInstanceUID"] + LABELS
            ],
            on="StudyInstanceUID",
            how="left",
            validate="one_to_one",
        )
    )

    e11_prob = aligned_e11[LABELS].to_numpy(float)

    groups = report_groups(train)
    gold_ids = np.flatnonzero(gold)

    # Five grouped splits over the gold subset. The parent OOF predictions are already
    # produced without using these labels, so this is a lightweight fusion-only validation.
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import roc_auc_score

    x_gold = train.iloc[gold_ids].copy()
    g_gold = groups[gold_ids]

    grid = (0.0, 0.10, 0.15, 0.20, 0.25, 0.30)
    chosen = {}

    for target_index, target in enumerate(LABELS):
        scores = {alpha: [] for alpha in grid}
        y_target = x_gold[target].to_numpy(float)

        if len(np.unique(y_target)) < 2:
            chosen[target] = 0.0
            continue

        for tr_rel, va_rel in GroupKFold(5).split(
            x_gold,
            y_target,
            groups=g_gold,
        ):
            y_va = y_target[va_rel]
            if len(np.unique(y_va)) < 2:
                continue

            base_rank = rank_columns(
                base_prob[gold_ids]
            )[:, target_index]
            e11_rank = rank_columns(
                e11_prob
            )[:, target_index]

            for alpha in grid:
                blended = (
                    (1.0 - alpha)
                    * base_rank[va_rel]
                    + alpha
                    * e11_rank[va_rel]
                )
                scores[alpha].append(
                    roc_auc_score(
                        y_va,
                        blended,
                    )
                )

        mean_scores = {
            alpha: (
                float(np.mean(values))
                if values
                else float("-inf")
            )
            for alpha, values in scores.items()
        }
        best_alpha = max(
            mean_scores,
            key=mean_scores.get,
        )

        if (
            best_alpha > 0.0
            and mean_scores[best_alpha]
            <= mean_scores[0.0] + 0.0005
        ):
            best_alpha = 0.0

        chosen[target] = float(best_alpha)

    return chosen


def _apply_e11_alpha(
    base_rank,
    e11_rank,
    alpha_map,
):
    out = np.asarray(
        base_rank,
        dtype=np.float64,
    ).copy()

    for j, target in enumerate(LABELS):
        alpha = float(
            alpha_map.get(target, 0.0)
        )
        out[:, j] = (
            (1.0 - alpha)
            * base_rank[:, j]
            + alpha
            * e11_rank[:, j]
        )

    return rank_columns(out)


In [ ]:

# ------------------------- final concordance -------------------------

RAD_BASE_ALPHA = 0.50
RAD_E11_FALLBACK = 0.20
RAD_PRESERVE = {"Baker's", "Fracture"}


def _e10_public_arm():
    """Three fat-suppressed RadImageNet slots, matching the audited E10 arm."""
    table = _rad_series_table()
    selected, sides = _rad_select(
        table,
        legacy_side=True,
    )
    encoder = _rad_encoder()
    heads, _ = _rad_public_heads()

    probabilities = _rad_family_predictions(
        encoder,
        heads,
        selected,
        sides,
        legacy=True,
        use_head_amp=True,
    )

    return {
        "rank": rank_columns(
            probabilities.mean(axis=0)
        ),
        "bank": np.stack([
            rank_columns(p)
            for p in probabilities
        ]),
    }


def _build_parent(d2, d3):
    return rank_columns(
        0.55 * d2["equal"]
        + 0.45 * d3["equal"]
    )


def _apply_e10(parent_rank, e10_rank, alpha=0.50):
    result = parent_rank.copy()
    for j, target in enumerate(LABELS):
        a = 0.0 if target in RAD_PRESERVE else float(alpha)
        result[:, j] = (
            (1.0 - a)
            * parent_rank[:, j]
            + a
            * e10_rank[:, j]
        )
    return rank_columns(result)


def _safe_rank_mix(*terms):
    total = sum(float(weight) for weight, _ in terms)
    if total <= 0:
        raise ValueError("rank mixture requires positive total weight")
    result = np.zeros_like(np.asarray(terms[0][1], dtype=np.float64))
    for weight, matrix in terms:
        result += (float(weight) / total) * np.asarray(matrix, dtype=np.float64)
    return rank_columns(result)


def _robust_rad(rank_bank):
    return _safe_rank_mix(
        (0.58, rank_bank.mean(axis=0)),
        (0.27, np.median(rank_bank, axis=0)),
        (0.15, robust_rank_consensus(rank_bank)),
    )


def run_final():
    # Always create a safe serialization before expensive optional arms.
    frame_from_scores(
        np.full(
            (len(TEST_IDS), len(LABELS)),
            0.5,
        ),
        "submission.csv",
    )

    d2 = run_dino2()
    d3 = run_dino3()

    parent = _build_parent(d2, d3)

    # Exact 0.914 anchor route: E10 at 0.50, Baker/Fracture preserved.
    e10 = _e10_public_arm()
    anchor = _apply_e10(
        parent,
        e10["rank"],
        RAD_BASE_ALPHA,
    )
    frame_from_scores(
        anchor,
        "submission_0914_anchor.csv",
    )

    # Robust E10 source: use fold consensus instead of pure probability mean.
    robust_e10 = _robust_rad(
        e10["bank"]
    )
    robust_anchor = _apply_e10(
        parent,
        robust_e10,
        RAD_BASE_ALPHA,
    )
    frame_from_scores(
        robust_anchor,
        "submission_e10_robust.csv",
    )

    # Third arm: diverse non-fat-suppressed + FS anchor.
    e11 = _e11_predict()

    e11_rank = e11["rank_bank"].mean(axis=0)
    e11_prob_rank = rank_columns(
        e11["probability"]
    )

    # Average-fold rank + probability mean gives a slightly different but very
    # low-variance representation of the same five trained heads.
    e11_consensus = _safe_rank_mix(
        (0.72, e11_rank),
        (0.28, e11_prob_rank),
    )

    # Fixed 0.20 residual reproduces the known direction.
    fixed_e11 = _apply_e11_alpha(
        anchor,
        e11_consensus,
        {
            target: (
                0.0
                if target in RAD_PRESERVE
                else RAD_E11_FALLBACK
            )
            for target in LABELS
        },
    )
    frame_from_scores(
        fixed_e11,
        "submission_e11_020.csv",
    )

    # OOF-selected target-specific residual.
    try:
        e11_alpha = _e11_choose_target_alpha()
    except Exception as _e11_err:
        print('E11 adaptive alpha failed; using fallback 0.20:', _e11_err)
        e11_alpha = {t: RAD_E11_FALLBACK for t in LABELS}

    for target in RAD_PRESERVE:
        e11_alpha[target] = 0.0

    adaptive_e11 = _apply_e11_alpha(
        anchor,
        e11_consensus,
        e11_alpha,
    )
    frame_from_scores(
        adaptive_e11,
        "submission_e11_adaptive.csv",
    )

    # Conservative shrinkage toward the known 0.914 anchor.
    safe = _safe_rank_mix(
        (0.72, anchor),
        (0.16, adaptive_e11),
        (0.12, robust_anchor),
    )

    # Main candidate: target-specific E11 residual plus a small
    # diversity contribution. The anchor remains the majority.
    main = _safe_rank_mix(
        (0.58, anchor),
        (0.25, adaptive_e11),
        (0.10, robust_anchor),
        (0.07, fixed_e11),
    )

    # More aggressive probe retained as a separate file.
    probe = _safe_rank_mix(
        (0.35, anchor),
        (0.40, adaptive_e11),
        (0.15, robust_anchor),
        (0.10, fixed_e11),
    )

    frame_from_scores(
        safe,
        "submission_safe.csv",
    )
    frame_from_scores(
        main,
        "submission_main.csv",
    )
    frame_from_scores(
        probe,
        "submission_probe.csv",
    )
    frame_from_scores(
        main,
        "submission.csv",
    )

    diagnostics = {
        "name": "RSNA Knee | DINOsaur V4",
        "anchor": "submission_0914_anchor.csv",
        "default": "submission_main.csv",
        "e11_alpha": e11_alpha,
        "e11_heads": int(e11["heads"]),
        "candidates": [
            "submission_0914_anchor.csv",
            "submission_e10_robust.csv",
            "submission_e11_020.csv",
            "submission_e11_adaptive.csv",
            "submission_safe.csv",
            "submission_main.csv",
            "submission_probe.csv",
        ],
    }
    (WORK / "dinosaur_v4_diagnostics.json").write_text(
        json.dumps(
            diagnostics,
            indent=2,
            sort_keys=True,
        ) + "\n"
    )

    print(
        "V4 E11 alpha map:",
        json.dumps(
            e11_alpha,
            indent=2,
            sort_keys=True,
        )
    )
    stamp("RSNA Knee | DINOsaur V4 complete")
    return main


FINAL = run_final()
